<div style="font-size: 24px; line-height: 1.6;">

# Misleading Variables: Some Columns Are Traps

## Cleanup is not housekeeping — it is deciding what evidence belongs

![Misleading Variables: Some Columns Are Traps](../images/Misleading_Variables.png)

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Takeaway

Functions introduced / reinforced: `pd.read_html`, `columns`, `dtypes`, `astype`, `to_datetime`, `select_dtypes`, `corr`, `pd.crosstab`, `drop`, `dropna`.

**Concept learned: not every column deserves to survive EDA.**

</div>

<div style="font-size: 24px; line-height: 1.6;">

### Imports

</div>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

<div style="font-size: 24px; line-height: 1.6;">

## The story

A churn dataset contains useful predictors (plan, tenure), identifiers (`customer_id`), dates, and variables that *happen after* churn (`refund_after_churn`). The latter look powerfully predictive, but they leak the answer.

</div>

<div style="font-size: 24px; line-height: 1.6; margin-top: 300px;">

## 1. Load the churn dataset

This file is an **HTML table** — the kind of thing you might scrape from a web page. `pd.read_html()` parses every `<table>` it finds and returns a **list of DataFrames**, so we take the first one with `[0]`.

</div>

In [2]:
churn = pd.read_html("../data/misleading_variables_churn.html")[0]
churn.head()

,customer_id,signup_date,age,tenure_months,plan,support_tickets,last_login_days_ago,refund_after_churn,churned
0,C00001,2020-11-22,70.0,12.1,free,0,6,False,0
1,C00002,2024-02-01,41.0,8.5,basic,1,6,False,0
2,C00003,2021-10-07,56.0,26.3,basic,3,14,False,0
3,C00004,2021-07-23,46.0,2.6,pro,0,4,False,0
4,C00005,2022-10-05,39.0,9.7,pro,0,4,False,0


In [3]:
churn.info()

<class 'pandas.DataFrame'>
RangeIndex: 3018 entries, 0 to 3017
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          3018 non-null   str    
 1   signup_date          3018 non-null   str    
 2   age                  2896 non-null   float64
 3   tenure_months        3018 non-null   float64
 4   plan                 3018 non-null   str    
 5   support_tickets      3018 non-null   int64  
 6   last_login_days_ago  3018 non-null   int64  
 7   refund_after_churn   3018 non-null   bool   
 8   churned              3018 non-null   int64  
dtypes: bool(1), float64(2), int64(3), str(3)
memory usage: 251.2 KB


<div style="font-size: 24px; line-height: 1.6; margin-top: 300px;">

## 2. Column audit with `df.columns`

Ask whether each column is an **identifier**, **outcome**, **predictor**, **date**, **proxy**, or **leak**.

</div>

In [4]:
list(churn.columns)

['customer_id',
 'signup_date',
 'age',
 'tenure_months',
 'plan',
 'support_tickets',
 'last_login_days_ago',
 'refund_after_churn',
 'churned']

<div style="font-size: 24px; line-height: 1.6; margin-top: 300px;">

## 3. Type audit with `df.dtypes`

Types reveal what loaded wrong: `signup_date` is still text, and `churned` came in as bare 0/1 integers. Both work mechanically and both hide meaning — the next two sections fix them.

</div>

In [5]:
churn.dtypes

customer_id                str
signup_date                str
age                    float64
tenure_months          float64
plan                       str
support_tickets          int64
last_login_days_ago      int64
refund_after_churn        bool
churned                  int64
dtype: object

<div style="font-size: 24px; line-height: 1.6; margin-top: 300px;">

## 4. Convert with `df.astype()`

`churned` loaded as 0/1 integers — fine for arithmetic, but a filter like `churn[churn["churned"]]` and every crosstab label read better when the column says what it means. `astype("bool")` makes the conversion, and the dtype output confirms it took.

</div>

In [6]:
churn["churned"] = churn["churned"].astype("bool")
churn["churned"].dtype

dtype('bool')

<div style="font-size: 24px; line-height: 1.6; margin-top: 300px;">

## 5. Convert dates with `pd.to_datetime()`

`pd.to_datetime()` turns date text into real datetime values — until then, 'dates' are just strings that sort alphabetically and cannot be compared, subtracted, or bucketed by year. The first payoff is immediate: min and max give the observation window, which you need for the leakage audit below (a column measured after this window cannot be a predictor).

</div>

In [7]:
churn["signup_date"] = pd.to_datetime(churn["signup_date"])
print(churn["signup_date"].dtype)
print("observation window:", churn["signup_date"].min().date(),
      "to", churn["signup_date"].max().date())

datetime64[us]
observation window: 2020-01-01 to 2024-02-08


<div style="font-size: 24px; line-height: 1.6; margin-top: 300px;">

## 6. Find suspicious correlations

A variable can look powerful because it leaks future information. We correlate every numeric column **and** the two booleans — cast to numbers, because `select_dtypes("number")` alone would silently drop them, hiding exactly the columns we suspect. Read down the `churned` column: two features stand out far above the honest predictors.

</div>

In [8]:
numeric = churn.select_dtypes(include=["number", "bool"]).astype(float)
numeric.corr().round(3)

,age,tenure_months,support_tickets,last_login_days_ago,refund_after_churn,churned
age,1.000,-0.003,0.052,0.145,0.119,0.176
tenure_months,-0.003,1.000,-0.015,-0.131,-0.094,-0.162
support_tickets,0.052,-0.015,1.000,0.153,0.104,0.182
last_login_days_ago,0.145,-0.131,0.153,1.000,0.497,0.833
refund_after_churn,0.119,-0.094,0.104,0.497,1.000,0.563
churned,0.176,-0.162,0.182,0.833,0.563,1.000


<div style="font-size: 24px; line-height: 1.6;">

Two numbers leap out of the `churned` column: `last_login_days_ago` at ≈ 0.83 and `refund_after_churn` at ≈ 0.56, while the honest predictors (age, tenure, tickets) all stay below 0.2. Ask of each: *could this value have been known at decision time, or is it a consequence of churn?* Days since last login is measured at extraction time — *after* churners stopped logging in — and a refund happens after churn. Both are consequences dressed as predictors; the next section confirms the refund case, and section 8 drops them both.

</div>

<div style="font-size: 24px; line-height: 1.6; margin-top: 300px;">

## 7. Cross-check a suspect with the outcome

`refund_after_churn` smells like a leak — by name alone.

</div>

In [9]:
pd.crosstab(churn["churned"], churn["refund_after_churn"])

refund_after_churn,False,True
churned,,
False,2547,0
True,304,167


<div style="font-size: 24px; line-height: 1.6;">

Read the crosstab one direction at a time. Most churners got no refund (304 of 471), so a missing refund proves nothing. But **every one of the 167 refunds belongs to a churner** — when the flag is True, churn is certain, because the refund only exists *after* the churn happened. That one-way perfection is the signature of a leak.

</div>

<div style="font-size: 24px; line-height: 1.6; margin-top: 300px;">

## 8. Drop columns with `df.drop()`

Dropping columns is an **analytical decision** that should be explained, not a default cleanup step.

</div>

In [10]:
safe = churn.drop(columns=["customer_id", "refund_after_churn", "last_login_days_ago"])
list(safe.columns)

['signup_date', 'age', 'tenure_months', 'plan', 'support_tickets', 'churned']

<div style="font-size: 24px; line-height: 1.6;">

Why each drop:

- `customer_id` — identifier, no predictive content
- `refund_after_churn` — happens after the outcome (**leak**)
- `last_login_days_ago` — measured at extraction time, may also leak

</div>

<div style="font-size: 24px; line-height: 1.6; margin-top: 300px;">

## 9. Drop rows vs. drop columns

Same verb, very different consequence. The row count drops by 122 here because that is how many customers have a missing `age` (check the non-null counts in `info()` above) — `dropna()` throws away whole customers to rescue one column, while dropping a column keeps every customer.

</div>

In [11]:
print("Drop rows with any NA:  ", churn.dropna().shape)
print("Drop one column:        ", churn.drop(columns=["customer_id"]).shape)

Drop rows with any NA:   (2896, 9)
Drop one column:         (3018, 8)


<div style="font-size: 24px; line-height: 1.6;">

## Discussion

- For each remaining column, when in the customer's lifetime is its value known? Before churn, at churn, or after?
- Which columns would you keep for an honest churn-prediction EDA, and which would you justify dropping in writing?

*Try to answer first. The answer is written in **white** — highlight the block below (drag your cursor across the empty space) to reveal it.*

<p style="color: #ffffff;"><strong>Answer.</strong> Keep a column only if its value already exists at the moment you would predict churn — tenure, plan type, usage-to-date. Anything known only <em>at</em> or <em>after</em> churn, like <code>refund_after_churn</code>, is leakage and should be dropped in writing with the reason: it inflates offline accuracy and then disappears in production, because at prediction time the churn has not happened yet.</p>

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Why this matters to an AI engineer

Leakage is the most common way a machine-learning project fails *silently*. A leaky feature makes the offline model look outstanding — every refund in this data belongs to a customer who already churned, so `refund_after_churn` hands the model a free, certain answer for a third of its churners — and then the model collapses in production, because at prediction time the churn has not happened yet and the feature's value does not exist.

No error message will ever tell you this. The offline metrics get *better* as the leak gets worse — which is exactly backwards. The only defense is the audit you just did: for every column, ask *when* its value becomes known, and drop — in writing — everything that is not knowable at prediction time.

</div>